In [2]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

In [5]:
## Load chunks
with open(r"..\data\chunked\chunks_type1.json") as f:
    chunks = json.load(f)

In [6]:
texts = [c["text"] for c in chunks]

In [8]:
## intialize model
model = SentenceTransformer("BAAI/bge-large-en")

c:\Users\ansul\OneDrive\Desktop\data science project\financial_rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ansul\.cache\huggingface\hub\models--BAAI--bge-large-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6068.76it/s]
BertModel

In [9]:
## generate embeddings
embeddings = model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.array(embeddings).astype("float32")

Batches: 100%|██████████| 39/39 [34:47<00:00, 53.53s/it]


In [13]:
embeddings.shape ## each chunk is represented by a 1024-dim vector

(610, 1024)

In [15]:
## build faiss index
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)  # cosine similarity
index.add(embeddings)

faiss.write_index(index, r"../data/embeddings/faiss_index_type1.index")

In [16]:
## Create metadata file
with open(r"../data/embeddings/metadata_type1.json", "w") as f:
    json.dump(chunks, f)

## Testing Retrival 

In [ ]:
## test retrieval
query = "What are the key financial metrics for the company?"
query_embedding = model.encode([query], normalize_embeddings=True).astype("float32")    
k = 5
D, I = index.search(query_embedding, k) 
print("Top 5 similar chunks:")
for idx in I[0]:
    print(f"- {chunks[idx]['text'][:200]}...")